# Chapter 4 — Remote Sensing Data & Geospatial Preprocessing

## Learning Objectives
- Load and inspect multi-band GeoTIFF files with rasterio
- Understand CRS (Coordinate Reference Systems) and spatial transforms
- Implement normalization strategies for multi-spectral imagery
- Compute spectral indices: NDVI, NDWI, NDBI, EVI
- Tile a large satellite scene into model-ready patches
- Build a reproducible preprocessing pipeline

## Estimated Duration: Theory 3h | Practical 3h | Total 6h
## Difficulty: Beginner → Intermediate

## Key EO Concepts
- GeoTIFF format: bands, nodata values, CRS, geotransform
- Coordinate Reference Systems: WGS84, UTM, EPSG codes
- Spatial resolution and its implications for deep learning
- Normalization: per-pixel vs per-band vs dataset-level
- Tiling strategy: size, overlap, border handling

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q torch rasterio rioxarray geopandas xarray matplotlib

import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_ROOT = Path('./data') if not IN_COLAB else Path('/content/data')
print(f'Device: {DEVICE}')

# Check geospatial libraries
try:
    import rasterio
    import geopandas as gpd
    import xarray as xr
    import rioxarray
    print(f'rasterio: {rasterio.__version__}')
    print(f'geopandas: {gpd.__version__}')
    print(f'xarray: {xr.__version__}')
except ImportError as e:
    print(f'Missing: {e} — run: pip install rasterio rioxarray geopandas xarray')

## 4.1 — Creating a Synthetic Sentinel-2 Scene for Demonstration

In a real project you would download a Sentinel-2 Level-2A tile from:
  - Copernicus Browser: https://browser.dataspace.copernicus.eu
  - sentinelsat Python package
  - Google Earth Engine

For this notebook, we create a synthetic 10-band GeoTIFF that mirrors
the structure of a real Sentinel-2 tile.

In [ ]:
import rasterio
from rasterio.transform import from_bounds
from rasterio.crs import CRS

# Create a synthetic Sentinel-2 scene (512×512 pixels, 10 bands)
H, W, C = 512, 512, 10
np.random.seed(42)

# Simulate different land cover zones
scene = np.zeros((H, W), dtype=np.int8)
scene[50:180, 50:180] = 1   # Urban (top-left)
scene[200:380, 50:250] = 2  # Forest (center-left)
scene[50:200, 250:450] = 3  # Agriculture (top-right)
scene[300:480, 300:480] = 4 # Water (bottom-right)

# Spectral signatures per land cover class (10 bands, ~SR × 10000)
spectral_sigs = {
    0: [500, 600, 550, 600, 700, 900, 1100, 1050, 900, 700],    # bare soil/background
    1: [800, 900, 1000, 1100, 1200, 1500, 1800, 1700, 1600, 1200],  # urban
    2: [200, 400, 300, 400, 1800, 3500, 4500, 4200, 2500, 1000],    # forest
    3: [300, 500, 400, 500, 1500, 3000, 3500, 3200, 2000, 900],     # agriculture
    4: [600, 700, 600, 500, 200, 100, 80, 50, 80, 60],              # water
}

# Build multi-band array
ms_scene = np.zeros((C, H, W), dtype=np.float32)
for cls_id, sig in spectral_sigs.items():
    mask = scene == cls_id
    for b, val in enumerate(sig):
        ms_scene[b][mask] = val + np.random.normal(0, val * 0.05, mask.sum())

# Save as GeoTIFF with proper CRS and geotransform
synthetic_path = DATA_ROOT / 'sentinel2_samples' / 'synthetic_s2_scene.tif'
synthetic_path.parent.mkdir(parents=True, exist_ok=True)

# Bounding box: somewhere in central Europe (EPSG:32632 = UTM zone 32N)
bbox = (600000, 5700000, 605120, 5705120)  # left, bottom, right, top (meters in UTM)
transform = from_bounds(*bbox, width=W, height=H)
crs = CRS.from_epsg(32632)  # UTM zone 32N

band_names = ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']

with rasterio.open(
    synthetic_path, 'w',
    driver='GTiff',
    height=H, width=W,
    count=C,
    dtype=np.float32,
    crs=crs,
    transform=transform,
    nodata=0,
) as dst:
    dst.write(ms_scene)
    # Tag bands with names
    for i, name in enumerate(band_names, 1):
        dst.update_tags(i, name=name)

print(f'Synthetic scene saved: {synthetic_path}')
print(f'Shape: {ms_scene.shape}  (bands × H × W)')
print(f'CRS: EPSG:32632 (WGS 84 / UTM zone 32N)')
print(f'Pixel size: {(bbox[2]-bbox[0])/W:.1f}m × {(bbox[3]-bbox[1])/H:.1f}m per pixel')

In [ ]:
# Inspect the GeoTIFF with rasterio
with rasterio.open(synthetic_path) as src:
    print('=== GeoTIFF Metadata ===')
    print(f'Driver: {src.driver}')
    print(f'Shape: {src.height} × {src.width} pixels')
    print(f'Bands: {src.count}')
    print(f'CRS: {src.crs}')
    print(f'Transform: {src.transform}')
    print(f'Bounding box: {src.bounds}')
    print(f'Resolution: {src.res} meters/pixel')
    print(f'Dtype: {src.dtypes}')
    print(f'Nodata: {src.nodata}')
    
    # Read all bands
    data = src.read()  # (C, H, W)
    print(f'\nData range per band:')
    for i, name in enumerate(band_names):
        print(f'  {name}: min={data[i].min():.0f}, max={data[i].max():.0f}, mean={data[i].mean():.0f}')

In [ ]:
# Visualise true colour and band composites
with rasterio.open(synthetic_path) as src:
    bands = src.read().astype(np.float32)

def percentile_stretch(arr, low=2, high=98):
    p_low = np.percentile(arr, low)
    p_high = np.percentile(arr, high)
    return np.clip((arr - p_low) / (p_high - p_low + 1e-8), 0, 1)

# Band mapping: B02=0, B03=1, B04=2, B08=6
rgb = np.stack([percentile_stretch(bands[2]),   # Red (B04)
                percentile_stretch(bands[1]),   # Green (B03)
                percentile_stretch(bands[0])],  # Blue (B02)
               axis=-1)

# False colour (NIR composite: NIR, Red, Green)
false_color = np.stack([percentile_stretch(bands[6]),  # NIR (B08)
                         percentile_stretch(bands[2]),  # Red (B04)
                         percentile_stretch(bands[1])], # Green (B03)
                        axis=-1)

# NDVI
nir = bands[6].astype(np.float64)
red = bands[2].astype(np.float64)
ndvi = (nir - red) / (nir + red + 1e-8)

# NDWI
green = bands[1].astype(np.float64)
ndwi = (green - nir) / (green + nir + 1e-8)

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
titles = ['True Colour (RGB)', 'False Colour (NIR+R+G)', 'NDVI', 'NDWI']
images = [rgb, false_color, ndvi, ndwi]
cmaps = [None, None, 'RdYlGn', 'RdBu']

for ax, img, title, cmap in zip(axes, images, titles, cmaps):
    if cmap:
        im = ax.imshow(img, cmap=cmap, vmin=-1, vmax=1)
        plt.colorbar(im, ax=ax, fraction=0.046)
    else:
        ax.imshow(img)
    ax.set_title(title, fontsize=11)
    ax.axis('off')

plt.suptitle('Synthetic Sentinel-2 Scene — Multiple Visualisations\n'
             '(Urban=top-left, Forest=center-left, Agriculture=top-right, Water=bottom-right)',
             fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Load with rioxarray for labelled array access
import rioxarray as rxr
import xarray as xr

da = rxr.open_rasterio(synthetic_path)
da['band'] = band_names  # label the band dimension
da.name = 'reflectance'

print('xarray DataArray:')
print(da)
print(f'\nDimensions: {dict(da.sizes)}')  # .sizes is a dim->size mapping; .dims is a tuple
print(f'CRS: {da.rio.crs}')
print(f'Resolution: {da.rio.resolution()}')

# Access by band name
ndvi_xr = (da.sel(band='B08') - da.sel(band='B04')) / (da.sel(band='B08') + da.sel(band='B04') + 1e-8)
ndvi_xr.name = 'NDVI'
print(f'\nNDVI stats: min={float(ndvi_xr.min()):.3f}, max={float(ndvi_xr.max()):.3f}')

## 4.2 — Normalization Strategies for Multi-Spectral EO

Three approaches, each with different trade-offs:

1. Per-image percentile stretch: [p2, p98] → [0, 1]
   - ✅ Robust to outliers (clouds, water bodies) 
   - ✅ Good for visualization
   - ⚠️  Statistics change per image → inconsistent across dataset

2. Dataset-level Z-score: (x - μ) / σ
   - ✅ Consistent across all images in the dataset
   - ✅ Allows pretrained model compatibility
   - ⚠️ Requires pre-computation of dataset statistics

3. Fixed physical range: x / 10000 (for SR scaled to 0–10000)
   - ✅ Physically meaningful (surface reflectance in 0–1 range)
   - ✅ No statistics needed
   - ⚠️ Does not centre the data (important for early conv layers)

In [ ]:
# Compare normalization strategies
bands_raw = bands.copy()  # (10, 512, 512)

# Strategy 1: Per-image percentile stretch
def normalize_percentile(arr, low=2, high=98):
    result = np.zeros_like(arr, dtype=np.float32)
    for i in range(arr.shape[0]):
        p_low = np.percentile(arr[i], low)
        p_high = np.percentile(arr[i], high)
        result[i] = np.clip((arr[i] - p_low) / (p_high - p_low + 1e-8), 0, 1)
    return result

# Strategy 2: Z-score with precomputed stats
# (In practice: computed from the full training set)
band_means = bands_raw.mean(axis=(1, 2))
band_stds = bands_raw.std(axis=(1, 2))
def normalize_zscore(arr, means, stds):
    return (arr - means[:, None, None]) / (stds[:, None, None] + 1e-8)

# Strategy 3: Fixed physical range
def normalize_physical(arr, max_val=10000):
    return np.clip(arr / max_val, 0, 1)

norm_pct = normalize_percentile(bands_raw)
norm_z = normalize_zscore(bands_raw, band_means, band_stds)
norm_phys = normalize_physical(bands_raw)

# Compare histograms for Band B04 (Red)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
norms = [norm_pct, norm_z, norm_phys]
titles = ['Percentile Stretch [0,1]', 'Z-score (mean=0, std≈1)', 'Physical Range ÷10000']
for ax, n, t in zip(axes, norms, titles):
    ax.hist(n[2].ravel(), bins=50, color='#2196F3', alpha=0.8, edgecolor='white')
    ax.set_title(f'{t}\nB04 (Red) distribution', fontsize=10)
    ax.set_xlabel('Normalized value')
    ax.set_ylabel('Count')
    ax.grid(alpha=0.3)

plt.suptitle('Normalization Strategy Comparison — Effect on Pixel Distribution', fontsize=11)
plt.tight_layout()
plt.show()

print('Recommendation for this course:')
print('  Classification (EuroSAT): use Z-score with precomputed dataset stats')
print('  Segmentation (Sentinel-2): use percentile stretch per-image for robustness')
print('  Transfer learning from ImageNet: use ImageNet mean/std for RGB bands')

## 4.3 — Scene Tiling for Deep Learning

A full Sentinel-2 tile is 10,980 × 10,980 pixels at 10m resolution.
This is far too large to feed into a CNN directly.

Solution: tile the scene into overlapping patches.

Tiling parameters:
  - tile_size = 512    → typical for segmentation (fits in 8GB VRAM)
  - overlap   = 64     → blending zone for prediction stitching
  - stride    = tile_size - overlap = 448

Edge tiles: pad with zeros OR reflect padding.

Overlap blending: use Gaussian weights at tile boundaries to avoid hard seams in the prediction map.

In [ ]:
# Demonstrate scene tiling
def tile_scene(image, tile_size=128, overlap=16):
    """Split (C, H, W) image into tiles. Returns tiles and their positions."""
    _, H, W = image.shape
    stride = tile_size - overlap
    tiles, positions = [], []
    
    row = 0
    while row < H:
        r0 = min(row, H - tile_size)
        if r0 < 0: r0 = 0
        col = 0
        while col < W:
            c0 = min(col, W - tile_size)
            if c0 < 0: c0 = 0
            tile = image[:, r0:r0+tile_size, c0:c0+tile_size]
            if tile.shape[1:] == (tile_size, tile_size):
                tiles.append(tile.copy())
                positions.append((r0, c0))
            col += stride
            if col >= W: break
        row += stride
        if row >= H: break
    
    return tiles, positions

# Tile our synthetic scene
TILE_SIZE, OVERLAP = 128, 16
tiles, positions = tile_scene(bands, tile_size=TILE_SIZE, overlap=OVERLAP)

print(f'Scene size: {bands.shape}')
print(f'Tile size: {TILE_SIZE}×{TILE_SIZE}, overlap: {OVERLAP}')
print(f'Stride: {TILE_SIZE - OVERLAP}')
print(f'Number of tiles: {len(tiles)}')

# Visualise tiling grid
fig, ax = plt.subplots(figsize=(8, 8))
rgb_display = np.stack([percentile_stretch(bands[2]), percentile_stretch(bands[1]), percentile_stretch(bands[0])], axis=-1)
ax.imshow(rgb_display)

import matplotlib.patches as mpatches
colors_tiles = plt.cm.tab20(np.linspace(0, 1, len(tiles)))
for (r0, c0), color in zip(positions, colors_tiles):
    rect = mpatches.Rectangle((c0, r0), TILE_SIZE, TILE_SIZE,
                               linewidth=1.5, edgecolor=color, facecolor='none', alpha=0.7)
    ax.add_patch(rect)

ax.set_title(f'Scene Tiling: {len(tiles)} tiles of {TILE_SIZE}×{TILE_SIZE} pixels\n'
             f'(overlap={OVERLAP}px, stride={TILE_SIZE-OVERLAP}px)', fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Complete preprocessing pipeline as a reusable class
class EOPreprocessingPipeline:
    """
    Reproducible preprocessing pipeline for Sentinel-2 scenes.
    
    Steps:
      1. Load GeoTIFF
      2. Select target bands
      3. Upsample low-resolution bands to target resolution
      4. Compute spectral indices
      5. Normalize
      6. Tile for model input
    """
    
    def __init__(self, tile_size=512, overlap=64, normalization='zscore'):
        self.tile_size = tile_size
        self.overlap = overlap
        self.normalization = normalization
        self.stats = None  # populated by fit()
    
    def fit(self, paths: list):
        """Compute normalization statistics from a list of scene paths."""
        all_means, all_stds = [], []
        for path in paths:
            with rasterio.open(path) as src:
                data = src.read().astype(np.float32)
            all_means.append(data.mean(axis=(1, 2)))
            all_stds.append(data.std(axis=(1, 2)))
        self.stats = {
            'mean': np.array(all_means).mean(axis=0),
            'std': np.array(all_stds).mean(axis=0),
        }
        return self
    
    def transform(self, path: Path) -> tuple:
        """Process a scene and return (tiles, positions, metadata)."""
        with rasterio.open(path) as src:
            data = src.read().astype(np.float32)
            meta = {'crs': src.crs, 'transform': src.transform,
                    'height': src.height, 'width': src.width}
        
        # Normalize
        if self.normalization == 'zscore' and self.stats is not None:
            data = (data - self.stats['mean'][:, None, None]) / (self.stats['std'][:, None, None] + 1e-8)
        elif self.normalization == 'percentile':
            data = normalize_percentile(data)
        else:
            data = data / 10000.0
        
        tiles, positions = tile_scene(data, self.tile_size, self.overlap)
        return tiles, positions, meta

# Demonstrate
pipeline = EOPreprocessingPipeline(tile_size=TILE_SIZE, overlap=OVERLAP, normalization='percentile')
tiles_norm, positions_norm, meta = pipeline.transform(synthetic_path)

print(f'Preprocessing pipeline output:')
print(f'  Tiles: {len(tiles_norm)} × {tiles_norm[0].shape}')
print(f'  Value range after normalization: [{tiles_norm[0].min():.3f}, {tiles_norm[0].max():.3f}]')
print(f'  CRS: {meta["crs"]}')
print(f'  Transform: {meta["transform"]}')

## Practical Exercises

### Exercise 4.1 — Real Data Download
Register at https://dataspace.copernicus.eu and download a small
Sentinel-2 Level-2A tile. Load it with rasterio and display the
true-colour composite.

### Exercise 4.2 — Spectral Index Comparison
Compute NDVI, NDWI, NDBI, and EVI for the synthetic scene.
Create a 2×2 subplot showing each index.
Which index best separates water from vegetation?

### Exercise 4.3 — Normalization Impact
Apply all three normalization strategies to tiles from the synthetic scene.
Feed them to the SimpleCNN from Chapter 3 (random init) and compare
the activation distributions after the first conv layer.

### Mini-Project 4
Build a complete preprocessing pipeline that:
  1. Loads a GeoTIFF
  2. Masks cloudy pixels (using a simple threshold on Band 9 / SCL band)
  3. Computes 3 spectral indices
  4. Normalizes all bands
  5. Tiles and saves as a PyTorch .pt dataset
  6. Verifies reproducibility (same output for same input)

In [ ]:
print('Chapter 4 Summary:')
print('  rasterio: read/write GeoTIFFs, access CRS and geotransform')
print('  rioxarray: labeled multi-band arrays with spatial metadata')
print('  Normalization: percentile for inference, Z-score for training')
print('  Tiling: stride = tile_size - overlap; edge tiles need padding')
print()
print('In Chapter 5: we build the first complete training pipeline')
print('using EuroSAT — from raw data to trained classifier with metrics.')